In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')


In [3]:
import os
from PIL import Image

# Define the base directory where your image dataset is located
# IMPORTANT: Update this path to your actual image dataset directory
base_dataset_path = '/content/drive/MyDrive/RiceLeafsDisease/train' # Example path

# Data structures to store results
image_stats = {
    'dimensions': {},
    'color_modes': {},
    'file_formats': {}
}
class_image_counts = {}
unopenable_files = []
image_paths_by_dimension = {} # New: To store file paths for each dimension

print(f"Starting image analysis for: {base_dataset_path}")

# Walk through the dataset directory
for root, dirs, files in os.walk(base_dataset_path):
    # Determine the class name based on the subdirectory name
    # Assuming direct subdirectories of base_dataset_path are classes
    relative_path = os.path.relpath(root, base_dataset_path)
    current_class = relative_path if relative_path != '.' else 'root'

    if current_class not in class_image_counts:
        class_image_counts[current_class] = 0

    for file in files:
        file_path = os.path.join(root, file)

        # Check for common image extensions
        if file.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp', '.tiff')):
            class_image_counts[current_class] += 1
            try:
                with Image.open(file_path) as img:
                    dimensions = img.size # (width, height)
                    color_mode = img.mode
                    file_format = img.format

                    # Collect dimensions stats
                    image_stats['dimensions'].setdefault(dimensions, 0)
                    image_stats['dimensions'][dimensions] += 1

                    # New: Store file path by dimension
                    image_paths_by_dimension.setdefault(dimensions, []).append(file_path)

                    # Collect color mode stats
                    image_stats['color_modes'].setdefault(color_mode, 0)
                    image_stats['color_modes'][color_mode] += 1

                    # Collect file format stats
                    image_stats['file_formats'].setdefault(file_format, 0)
                    image_stats['file_formats'][file_format] += 1

            except IOError: # Catch errors for files that cannot be opened
                unopenable_files.append(file_path)
                class_image_counts[current_class] -= 1 # Decrement if it couldn't be opened

print("\n--- Analysis Complete ---")

Starting image analysis for: /content/drive/MyDrive/RiceLeafsDisease/train

--- Analysis Complete ---


In [4]:
# Display Results

print("\n### Image Dimensions Distribution:")
for dim, count in sorted(image_stats['dimensions'].items()):
    print(f"  {dim[0]}x{dim[1]}: {count} images")

print("\n### Color Mode Distribution:")
for mode, count in sorted(image_stats['color_modes'].items()):
    print(f"  {mode}: {count} images")

print("\n### File Format Distribution:")
for fmt, count in sorted(image_stats['file_formats'].items()):
    print(f"  {fmt}: {count} images")

print("\n### Number of Images in Each Class:")
for class_name, count in class_image_counts.items():
    print(f"  Class '{class_name}': {count} images")

if unopenable_files:
    print("\n### Unopenable Files:")
    for f in unopenable_files:
        print(f"  - {f}")
else:
    print("\nAll image files were successfully opened.")

# New: Display specific images with dimension 1600x1548
target_dimension = (1600, 1548)
if target_dimension in image_paths_by_dimension:
    print(f"\n### Files with dimension {target_dimension[0]}x{target_dimension[1]}:")
    for path in image_paths_by_dimension[target_dimension]:
        print(f"  - {path}")
else:
    print(f"\nNo files found with dimension {target_dimension[0]}x{target_dimension[1]}.")


### Image Dimensions Distribution:
  1600x1548: 2 images
  1600x1600: 2098 images

### Color Mode Distribution:
  RGB: 2100 images

### File Format Distribution:
  JPEG: 2100 images

### Number of Images in Each Class:
  Class 'root': 0 images
  Class 'leaf_blast': 350 images
  Class 'bacterial_leaf_blight': 350 images
  Class 'brown_spot': 350 images
  Class 'leaf_scald': 350 images
  Class 'healthy': 350 images
  Class 'narrow_brown_spot': 350 images

All image files were successfully opened.

### Files with dimension 1600x1548:
  - /content/drive/MyDrive/RiceLeafsDisease/train/narrow_brown_spot/narrow_brown (297).jpg
  - /content/drive/MyDrive/RiceLeafsDisease/train/narrow_brown_spot/narrow_brown (287).jpg


In [5]:
from pathlib import Path
import hashlib
from collections import defaultdict

# Duplicate detection

dataset_path = Path("/content/drive/MyDrive/RiceLeafsDisease/train")

hashes = {}  # Stores {hash: first_seen_path}
duplicate_files = defaultdict(list)  # Stores {class_name: [list_of_duplicate_file_paths_in_that_class]}
total_unique_images = 0

print(f"Detecting duplicate images in: {dataset_path}")

for file_path in dataset_path.rglob("*"):
    if file_path.is_file() and file_path.suffix.lower() in [".jpg", ".jpeg"]:
        try:
            file_hash = hashlib.md5(file_path.read_bytes()).hexdigest()

            if file_hash in hashes:
                # This file_path is a duplicate of an earlier file
                class_name = file_path.parent.name
                duplicate_files[class_name].append(str(file_path))
            else:
                hashes[file_hash] = file_path
                total_unique_images += 1  # Count only the first instance as unique

        except Exception as e:
            print(f"Could not process {file_path}: {e}")

print("\n--- Duplicate Detection Complete ---\n")

# Report duplicates
total_duplicates_found_count = 0

if duplicate_files:
    print("### Duplicate Images Summary ###")
    for class_name, files_in_class in sorted(duplicate_files.items()):
        if files_in_class:
            num_duplicates_in_class = len(files_in_class)
            total_duplicates_found_count += num_duplicates_in_class
            print(f"Class '{class_name}' has {num_duplicates_in_class} duplicate image(s):")
            for f_path in files_in_class:
                print(f"  - {Path(f_path).name}")  # Just print the image name
    print(f"\nTotal number of duplicate image files found: {total_duplicates_found_count}")
else:
    print("No duplicate images found.")

print(f"Total unique images in dataset: {total_unique_images}")

Detecting duplicate images in: /content/drive/MyDrive/RiceLeafsDisease/train

--- Duplicate Detection Complete ---

### Duplicate Images Summary ###
Class 'bacterial_leaf_blight' has 53 duplicate image(s):
  - bacterial_leaf_blight (317).JPG
  - bacterial_leaf_blight (3).JPG
  - bacterial_leaf_blight (326).JPG
  - bacterial_leaf_blight (301).JPG
  - bacterial_leaf_blight (331).JPG
  - bacterial_leaf_blight (300).JPG
  - bacterial_leaf_blight (299).jpg
  - bacterial_leaf_blight (33).jpg
  - bacterial_leaf_blight (293).JPG
  - bacterial_leaf_blight (318).JPG
  - bacterial_leaf_blight (297).JPG
  - bacterial_leaf_blight (325).JPG
  - bacterial_leaf_blight (329).JPG
  - bacterial_leaf_blight (343).JPG
  - bacterial_leaf_blight (347).JPG
  - bacterial_leaf_blight (68).JPG
  - bacterial_leaf_blight (88).JPG
  - bacterial_leaf_blight (90).JPG
  - bacterial_leaf_blight (337).jpg
  - bacterial_leaf_blight (349).JPG
  - bacterial_leaf_blight (89).JPG
  - bacterial_leaf_blight (6).JPG
  - bacteri

### Checking for Duplicates Between Training and Testing Datasets

It's crucial to ensure there's no data leakage between your training and testing datasets. This script will help identify any identical images present in both sets.

In [6]:
from pathlib import Path
import hashlib

# Define the paths for your training and testing datasets
# IMPORTANT: Update these paths to your actual dataset directories
train_dataset_path = Path('/content/drive/MyDrive/RiceLeafsDisease/train')
test_dataset_path = Path('/content/drive/MyDrive/RiceLeafsDisease/validation') # Assuming a 'test' folder exists at the same level

# Dictionaries to store hashes and their corresponding file paths
train_hashes = {}
test_hashes = {}

print(f"Processing training dataset: {train_dataset_path}")
for file_path in train_dataset_path.rglob("*"):
    if file_path.is_file() and file_path.suffix.lower() in [".jpg", ".jpeg"]:
        try:
            file_hash = hashlib.md5(file_path.read_bytes()).hexdigest()
            train_hashes[file_hash] = file_path
        except Exception as e:
            print(f"Could not process training image {file_path}: {e}")

print(f"\nProcessing testing dataset: {test_dataset_path}")
for file_path in test_dataset_path.rglob("*"):
    if file_path.is_file() and file_path.suffix.lower() in [".jpg", ".jpeg"]:
        try:
            file_hash = hashlib.md5(file_path.read_bytes()).hexdigest()
            test_hashes[file_hash] = file_path
        except Exception as e:
            print(f"Could not process testing image {file_path}: {e}")

print("\n--- Cross-Dataset Duplicate Detection Complete ---\n")

# Find common hashes (duplicates) between train and test sets
duplicate_in_both = set(train_hashes.keys()) & set(test_hashes.keys())

if duplicate_in_both:
    print(f"### Found {len(duplicate_in_both)} duplicate images in both training and testing datasets: ###")
    for common_hash in duplicate_in_both:
        train_file = train_hashes[common_hash]
        test_file = test_hashes[common_hash]
        print(f"  - Train: {train_file.relative_to(train_dataset_path)}")
        print(f"  - Test:  {test_file.relative_to(test_dataset_path)}")
else:
    print("No duplicate images found between the training and testing datasets. Good job!")


Processing training dataset: /content/drive/MyDrive/RiceLeafsDisease/train

Processing testing dataset: /content/drive/MyDrive/RiceLeafsDisease/validation

--- Cross-Dataset Duplicate Detection Complete ---

### Found 83 duplicate images in both training and testing datasets: ###
  - Train: bacterial_leaf_blight/bacterial_leaf_blight (324).JPG
  - Test:  bacterial_leaf_blight/bacterial_val (78).JPG
  - Train: brown_spot/brown_spot (112).jpg
  - Test:  brown_spot/brown_val (84).jpg
  - Train: narrow_brown_spot/narrow_brown (8).jpg
  - Test:  narrow_brown_spot/narrow_brown_val (7).jpg
  - Train: bacterial_leaf_blight/bacterial_leaf_blight (105).JPG
  - Test:  bacterial_leaf_blight/bacterial_val (14).JPG
  - Train: bacterial_leaf_blight/bacterial_leaf_blight (99).JPG
  - Test:  bacterial_leaf_blight/bacterial_val (12).JPG
  - Train: leaf_blast/leaf_blast (195).jpg
  - Test:  leaf_blast/leaf_blast_val (58).jpg
  - Train: leaf_scald/leaf_scald (204).jpg
  - Test:  leaf_scald/leaf_scald_val 

In [16]:
from pathlib import Path
import hashlib
from collections import defaultdict

# Initialize data structures for test duplicate detection
test_hashes_internal = {}
duplicate_test_files = defaultdict(list)

print("\n### Detecting duplicate images within the TEST dataset ###")

# Iterate through X_test (image paths for the test set)
for img_path_obj in X_test:
    try:
        # Calculate MD5 hash of the image file content
        with open(img_path_obj, "rb") as f:
            file_hash = hashlib.md5(f.read()).hexdigest()

        if file_hash in test_hashes_internal:
            # This file is a duplicate of an earlier file in the test set
            class_name = img_path_obj.parent.name
            duplicating_path_name = img_path_obj.name # Just the file name
            duplicate_test_files[class_name].append(duplicating_path_name)
        else:
            test_hashes_internal[file_hash] = img_path_obj

    except Exception as e:
        print(f"Could not process test image {img_path_obj}: {e}")

print("\n--- Test Duplicate Detection Complete ---\n")

# Report duplicates found in the test set
total_test_duplicates_found = 0

if duplicate_test_files:
    print("### Duplicate Images within Test Set Summary ###")
    for class_name, files_in_class in sorted(duplicate_test_files.items()):
        if files_in_class:
            num_duplicates_in_class = len(files_in_class)
            total_test_duplicates_found += num_duplicates_in_class
            print(f"Class '{class_name}' has {num_duplicates_in_class} duplicate image(s):")
            for f_name in files_in_class:
                print(f"  - {f_name}")
    print(f"\nTotal number of duplicate image files found in test set: {total_test_duplicates_found}")
else:
    print("No duplicate images found within the test dataset.")



### Detecting duplicate images within the TEST dataset ###

--- Test Duplicate Detection Complete ---

No duplicate images found within the test dataset.


In [7]:
print('\n### Number of Duplicate Images Between Train and Test, by Class ###')

duplicates_per_class = defaultdict(int)
for common_hash in duplicate_in_both:
    train_file_path = train_hashes[common_hash]
    # Extract class name from the training file path
    class_name = train_file_path.parent.name
    duplicates_per_class[class_name] += 1

if duplicates_per_class:
    for class_name, count in sorted(duplicates_per_class.items()):
        print(f"  Class '{class_name}': {count} duplicates")
else:
    print("No duplicate images found between training and testing datasets.")


### Number of Duplicate Images Between Train and Test, by Class ###
  Class 'bacterial_leaf_blight': 20 duplicates
  Class 'brown_spot': 15 duplicates
  Class 'leaf_blast': 2 duplicates
  Class 'leaf_scald': 37 duplicates
  Class 'narrow_brown_spot': 9 duplicates


In [8]:
print('### Overlap of Duplicates ###')

# Get a set of all duplicate file paths within the training set (from the previous analysis)
intra_train_duplicate_paths = set()
for class_name, paths in duplicate_files.items():
    for p in paths:
        intra_train_duplicate_paths.add(Path(p))

# Get a set of all training file paths that are duplicates in both train and test sets
inter_dataset_train_duplicate_paths = set()
for common_hash in duplicate_in_both:
    inter_dataset_train_duplicate_paths.add(train_hashes[common_hash])

# Find the overlap
overlap_duplicates = intra_train_duplicate_paths.intersection(inter_dataset_train_duplicate_paths)

print(f"Total intra-train duplicates identified: {len(intra_train_duplicate_paths)}")
print(f"Total train images found in both train and test sets: {len(inter_dataset_train_duplicate_paths)}")
print(f"Number of images that are both intra-train duplicates AND inter-dataset duplicates (from the train set): {len(overlap_duplicates)}")

if overlap_duplicates:
    print("\nExample overlapping duplicate files (from training set):")
    for i, path in enumerate(list(overlap_duplicates)[:5]): # Display up to 5 examples
        print(f"  - {path.relative_to(train_dataset_path)}")
    if len(overlap_duplicates) > 5:
        print(f"  ... and {len(overlap_duplicates) - 5} more.")
else:
    print("No overlap found between intra-train duplicates and inter-dataset duplicates.")

### Overlap of Duplicates ###
Total intra-train duplicates identified: 159
Total train images found in both train and test sets: 83
Number of images that are both intra-train duplicates AND inter-dataset duplicates (from the train set): 0
No overlap found between intra-train duplicates and inter-dataset duplicates.


In [9]:
from collections import defaultdict

print('\n### Unique Images Remaining per Class ###')
print("Note: The 'root' class likely contains images directly in the base_dataset_path, not in subfolders.")

# Get all unique hashes after intra-train duplicate detection for each class.
class_unique_hashes = defaultdict(set)
for file_hash, file_path_obj in hashes.items(): # `hashes` stores only the first seen unique path for each hash.
    # Determine the class name. If file_path_obj is directly in train_dataset_path,
    # its parent name will be the same as train_dataset_path.name (e.g., 'train').
    # In this case, we consider it 'root' as per previous logic.
    class_name_from_path = file_path_obj.parent.name

    # If the file's parent path is exactly the train_dataset_path, it's in the 'root' class.
    if str(file_path_obj.parent) == str(train_dataset_path):
        class_unique_hashes['root'].add(file_hash)
    else:
        class_unique_hashes[class_name_from_path].add(file_hash)

# Now, count the unique images per class that are NOT in duplicate_in_both.
final_unique_counts = defaultdict(int)
for class_name_from_hash, hash_set in class_unique_hashes.items():
    for h in hash_set:
        if h not in duplicate_in_both: # Only count if it's not a duplicate in both train/test
            final_unique_counts[class_name_from_hash] += 1

# Print the final unique counts
for class_name, count in sorted(final_unique_counts.items()):
    print(f"  Class '{class_name}': {count} unique images")



### Unique Images Remaining per Class ###
Note: The 'root' class likely contains images directly in the base_dataset_path, not in subfolders.
  Class 'bacterial_leaf_blight': 277 unique images
  Class 'brown_spot': 305 unique images
  Class 'healthy': 348 unique images
  Class 'leaf_blast': 335 unique images
  Class 'leaf_scald': 271 unique images
  Class 'narrow_brown_spot': 322 unique images


# Dataset Cleaning and Splitting

- Identify and handle duplicate images.
- Prevent any image leakage between train, validation, and test sets.
- Create a clean, stratified dataset split.
- Preserve the original dataset without modification.
- Verify the final split and class distribution.
- Keep the test set completely unseen for final model evaluation.
- Perform image preprocessing and augmentation only after the clean split.

In [11]:
from pathlib import Path
import hashlib
import shutil
from collections import defaultdict
from sklearn.model_selection import train_test_split


source_path = Path("/content/drive/MyDrive/RiceLeafsDisease/train")
output_path = Path("/content/drive/MyDrive/RiceLeafsDisease/clean_dataset")


# Group images by their actual file content so exact duplicates are treated as one image
image_groups = defaultdict(list)

for class_dir in source_path.iterdir():
    if not class_dir.is_dir():
        continue

    for img_path in class_dir.iterdir():
        if img_path.suffix.lower() in [".jpg", ".jpeg", ".png"]:
            with open(img_path, "rb") as f:
                image_hash = hashlib.md5(f.read()).hexdigest()

            image_groups[image_hash].append(img_path)


# Keep one representative from each duplicate group
unique_images = [
    (files[0], files[0].parent.name)
    for files in image_groups.values()
]

image_paths = [item[0] for item in unique_images]
labels = [item[1] for item in unique_images]


# Stratification keeps the proportion of the six classes consistent in both sets
X_train, X_test, y_train, y_test = train_test_split(
    image_paths,
    labels,
    test_size=0.20,
    random_state=42,
    stratify=labels
)


# Create the new dataset structure without touching the original dataset
class_names = sorted(set(labels))

for split in ["train", "test"]:
    for class_name in class_names:
        (output_path / split / class_name).mkdir(
            parents=True,
            exist_ok=True
        )


# Copy the selected images into their respective splits
for img_path, class_name in zip(X_train, y_train):
    shutil.copy2(
        img_path,
        output_path / "train" / class_name / img_path.name
    )

for img_path, class_name in zip(X_test, y_test):
    shutil.copy2(
        img_path,
        output_path / "test" / class_name / img_path.name
    )


print(f"Unique images : {len(unique_images)}")
print(f"Train images  : {len(X_train)}")
print(f"Test images   : {len(X_test)}")

Unique images : 1941
Train images  : 1552
Test images   : 389


### Step 1.2: Check Class Distribution

- Check the number of images from each class in train and test.
- Confirm that stratification preserved the class proportions.
- Do not create or modify any files yet.

In [13]:
from collections import Counter


# Compare class counts in the two splits
train_counts = Counter(y_train)
test_counts = Counter(y_test)

for class_name in sorted(train_counts):
    print(
        f"{class_name:<25} "
        f"Train: {train_counts[class_name]:>3} | "
        f"Test: {test_counts[class_name]:>3}"
    )

bacterial_leaf_blight     Train: 238 | Test:  59
brown_spot                Train: 256 | Test:  64
healthy                   Train: 278 | Test:  70
leaf_blast                Train: 269 | Test:  68
leaf_scald                Train: 246 | Test:  62
narrow_brown_spot         Train: 265 | Test:  66


### Step 1.3: Create Clean Dataset Structure

- Create separate train and test directories.
- Create one folder for each of the six classes.
- Keep the original dataset unchanged.
- No images are copied yet.

In [14]:
from pathlib import Path

output_path = Path("/content/drive/MyDrive/RiceLeafsDisease/clean_dataset")

# Create the train/test structure using the classes already present in the split
for split in ["train", "test"]:
    for class_name in sorted(set(labels)):
        (output_path / split / class_name).mkdir(
            parents=True,
            exist_ok=True
        )

print("Dataset structure created at:")
print(output_path)

Dataset structure created at:
/content/drive/MyDrive/RiceLeafsDisease/clean_dataset


### Step 1.4: Copy Images to the Clean Dataset

- Copy the selected training images into the new train folders.
- Copy the selected testing images into the new test folders.
- Keep the original dataset unchanged.

In [15]:
import shutil

# Copy the images according to the split we already created
for img_path, class_name in zip(X_train, y_train):
    destination = output_path / "train" / class_name / img_path.name
    shutil.copy2(img_path, destination)

for img_path, class_name in zip(X_test, y_test):
    destination = output_path / "test" / class_name / img_path.name
    shutil.copy2(img_path, destination)

print("Images copied successfully.")

Images copied successfully.


### Step 1.5: Verify the Clean Dataset

- Count the images actually present in each train and test folder.
- Confirm the copied dataset matches the planned split.
- Confirm the total number of images is 1,941.

In [17]:
from pathlib import Path

# Count the actual image files present in each class folder
for split in ["train", "test"]:
    print(f"\n{split.upper()}")

    total = 0

    for class_dir in sorted((output_path / split).iterdir()):
        count = sum(
            1 for file in class_dir.iterdir()
            if file.suffix.lower() in [".jpg", ".jpeg", ".png"]
        )

        total += count
        print(f"{class_dir.name:<25} {count}")

    print(f"Total: {total}")


TRAIN
bacterial_leaf_blight     238
brown_spot                256
healthy                   278
leaf_blast                269
leaf_scald                246
narrow_brown_spot         265
Total: 1552

TEST
bacterial_leaf_blight     59
brown_spot                64
healthy                   70
leaf_blast                68
leaf_scald                62
narrow_brown_spot         66
Total: 389
